# Network Analysis with External APIs


In the previous section, we performed network analysis using a locally built graph and the `networkx` library.

In this section, we will look at an alternative approach — **external routing API services**. The computations are handled server-side, and we retrieve the results on demand.

To work with these services, it helps to understand how APIs are structured: how requests are formed, what parameters they accept, and how to handle the responses.

We begin with the basic principles of working with APIs, then walk through three routing services: **OSRM**, **OpenRouteService**, and **GraphHopper**.


## 0. Importing Libraries


In [ ]:
import requests
import polyline
import geopandas as gpd

from shapely.geometry import LineString

- [**Requests**](https://docs.python-requests.org/) (`requests`) — a Python library for sending HTTP requests and working with web services (APIs). You use it to retrieve data from servers (GET), submit data (POST), and handle responses, JSON included. It simplifies interaction with external APIs and is widely used for fetching and exchanging data over the internet.

- [**polyline**](https://pypi.org/project/polyline/) (`polyline`) — a small library for decoding the encoded polyline format that many routing services use to compress route geometry. We use it below to turn an API response into a list of coordinates.

## 1. Application Programming Interface (API)


An **API (Application Programming Interface)** is how one program talks to another.

For our purposes it is a web service: we send it a request with some parameters — a pair of coordinates, say — and it sends back data, such as a route or a distance.

That exchange goes over an **HTTP request**, the standard way a client (our program) and a server pass data between them.


### 1.1. How an API Request is Structured

An API request usually has three parts:

- a **URL** — the address of the service (e.g. `https://api.service.com/route`);
- **parameters** — the input data (coordinates, transport mode, etc.);
- **headers** — additional metadata (e.g. an API key or the expected response format).


### 1.2. Request Methods

HTTP has several request methods, each for a different kind of operation:

- **GET** — retrieves data from the server. Parameters are passed in the URL.

- **POST** — sends data to the server. Parameters are passed in the request body. Commonly used for computations.

- **PUT** — fully replaces an existing resource on the server.

- **PATCH** — partially updates a resource (modifies specific fields only).

- **DELETE** — removes a resource from the server.

We will mostly use **GET** and **POST**. Which one an operation needs is always in the service's documentation, so read it carefully.


### 1.3. Reading the Documentation

Every API has documentation that describes:

- the available endpoints (e.g. routing, matrix, isochrones);
- the required parameters;
- the request method to use (GET, POST, etc.);
- the format of the response.

Many services require an **API key**, a string that identifies you to the service. It is how they control access, enforce rate limits and keep track of usage.

### 1.4. The `requests` Library

We send our HTTP requests from Python with the `requests` library.

The server answers with a response that carries:

- a **status code** — e.g. `200` for success, `400` for a bad request, `401` for an authorisation error;
- a **response body** — the data, most commonly in JSON format;
- **response headers** — additional information about the result.

### 1.5. Checking the Response Status

After sending a request, confirm that the server processed it successfully.

The `.status_code` attribute holds the **HTTP status code**.

The most common codes are:

- `200` — the request succeeded; proceed to handling the response;
- `4xx` — a client-side error (e.g. invalid URL, bad parameters, or an incorrect API key);
- `5xx` — a server-side error.

If the code is **not `200`**, do not parse the result — read the response body instead. Between the code and the body, the reason is usually obvious.

_Note: many routing services offer ready-made Python client libraries, but in this section we work directly with the API to develop a solid understanding of how requests work._


## 2. APIs for Network Analysis


In this section, we will work with several popular APIs and use them to solve common network analysis tasks:

- finding shortest routes;
- retrieving distances and travel times;
- generating isochrones;
- computing distance matrices.

Each service has its own request format, set of features, and rate limits — all described in its documentation. For each service, we will work through one or more examples, walking through the full workflow from forming a request to processing the response and visualising the result.


### 2.1. Open Source Routing Machine (OSRM)

**OSRM (Open Source Routing Machine)** is a fast, open-source routing engine built on OpenStreetMap data.

OSRM does not require an API key — requests can be sent directly.

[Documentation](https://project-osrm.org/docs/v5.24.0/api/)


#### 2.1.1. Shortest Route

The OSRM documentation describes how to request a route between two points.

The endpoint is:

```
/route/v1/{profile}/{coordinates}
```

where:

- `profile` — the transport mode (e.g. `driving`);
- `coordinates` — a list of points in `longitude,latitude` format, separated by `;`.

Additional query parameters can also be passed, for example `overview=full` to include the full route geometry in the response.


##### 2.1.1.1. Building the Request URL

Build the full URL for the API call.
We define the start and end coordinates and format them as the service expects.


In [ ]:
base_url_osrm = "https://router.project-osrm.org/route/v1/driving/"

# Coordinates (longitude, latitude)
start_coords_osrm = [16.3725, 48.2082]   # Stephansplatz
end_coords_osrm = [16.3122, 48.1847]     # Schönbrunn Palace

# Format coordinates as strings
start_osrm = f"{start_coords_osrm[0]},{start_coords_osrm[1]}"
end_osrm = f"{end_coords_osrm[0]},{end_coords_osrm[1]}"

# Build the full URL
url_osrm = f"{base_url_osrm}{start_osrm};{end_osrm}?overview=full"

> **Important:** different APIs use different coordinate ordering.
> Some services expect longitude, latitude; others expect latitude, longitude. Always check the documentation before sending a request.


##### 2.1.1.2. Sending the Request


Send the HTTP request with the `requests` library.


In [ ]:
response_osrm = requests.get(url_osrm)

##### 2.1.1.3. Checking the Response Status


Before processing the response, let's confirm that the request succeeded.


In [ ]:
response_osrm.status_code

If the status is `200`, we can proceed to parse the result. Any other code indicates an error — inspect the response body to understand what went wrong.


##### 2.1.1.4. Parsing the Response

The server returns a JSON response, which can be converted to a Python dictionary using the `.json()` method.


In [ ]:
data_osrm = response_osrm.json()

The top-level keys show how the response is organised. A full description of the response schema is available in the API documentation.


In [ ]:
data_osrm.keys()

Route information is stored under the `routes` key, which is a list of route objects.
Since we requested a single route, we access the first element and examine its structure.


In [ ]:
route_osrm = data_osrm["routes"][0]

route_osrm.keys()

Pull the key route properties out of the response.


In [ ]:
distance_osrm = route_osrm["distance"]
duration_osrm = route_osrm["duration"]
geometry_osrm = route_osrm["geometry"]

print(f"Distance (metres): {distance_osrm}")
print(f"Duration (seconds): {duration_osrm}")
print(f"Geometry: {geometry_osrm[:50]}...")

The route geometry is returned in encoded polyline format to reduce the size of the response.
To use it further (e.g. for visualisation), it must be decoded into a list of coordinates.


##### 2.1.1.5. Decoding the Geometry

Decode the route geometry from the encoded polyline format into a list of coordinates with the `polyline` library.


In [ ]:
decoded_route_osrm = polyline.decode(geometry_osrm)

The result is a list of coordinate pairs describing the route.
Since there may be many points, let's print just the first few:


In [ ]:
decoded_route_osrm[:5]

Each point is returned as a (latitude, longitude) pair, since `polyline.decode()` uses that order.

However, `shapely` expects coordinates in (longitude, latitude) order, so we need to swap them before creating the geometry.


In [ ]:
route_line_osrm = LineString([(lon, lat) for lat, lon in decoded_route_osrm])

##### 2.1.1.6. Creating a GeoDataFrame

Wrap the line in a `GeoDataFrame` so it can go on a map.


In [ ]:
route_gdf_osrm = gpd.GeoDataFrame(
    {"name": ["Route"]},
    geometry=[route_line_osrm],
    crs="EPSG:4326"
)

The same for the start and end points:


In [ ]:
points_gdf_osrm = gpd.GeoDataFrame(
    {"name": ["Start", "End"]},
    geometry=gpd.points_from_xy(
        [start_coords_osrm[0], end_coords_osrm[0]], 
        [start_coords_osrm[1], end_coords_osrm[1]]
    ),
    crs="EPSG:4326"
)

##### 2.1.1.7. Visualising the Result

And the route with its endpoints on an interactive map.


In [ ]:
m = route_gdf_osrm.explore(
    tiles="cartodbpositron",
    color="#FCDD9D",
    style_kwds={"weight": 5},
    tooltip="name"
)

points_gdf_osrm.explore(
    m=m,
    color="#A3B565",
    marker_kwds={"radius": 6},
    tooltip="name"
)

In this example, we covered the basic OSRM workflow — retrieving a route between two points. The service also supports additional capabilities such as distance matrices; these are described in the documentation.


### 2.2. OpenRouteService

**OpenRouteService (ORS)** is a routing and spatial analysis service built on OpenStreetMap data.
It supports routing, isochrone generation, and distance matrix calculation.

OpenRouteService requires an **API key**, which you can obtain by registering on the service website.

[Documentation](https://openrouteservice.org/dev/#/api-docs)

> **Note:** the cells in this section will not produce output automatically, as they require a personal API key. You can obtain one from your OpenRouteService dashboard and run the code locally. The placeholder for the key is marked as `your_ors_key`.


#### 2.2.0. Storing the API Key

Before getting started, obtain an API key from [**OpenRouteService**](https://openrouteservice.org) by registering on the site and copying the key from your dashboard.

Store the key in a variable so it can be used in API requests.


In [ ]:
ors_api_key = "your_ors_key"

#### 2.2.1. Shortest Route

OpenRouteService can also return a route between two points.

The endpoint is:

```python
/v2/directions/{profile}
```

where:

- `profile` — the transport mode (e.g. `driving-car`);
- the start and end points are passed as query parameters.


##### 2.2.1.1. Building the Request Parameters

The endpoint for route calculation:


In [ ]:
url_directions_ors = "https://api.openrouteservice.org/v2/directions/driving-car"

Now let's define the start and end coordinates.


In [ ]:
# Coordinates (longitude, latitude)
start_coords_ors = [16.3956, 48.2166]    # Wiener Riesenrad
end_coords_ors = [16.3806, 48.1917]      # Belvedere

# Build the query parameters from the coordinates
params_directions_ors = {
    "start": f"{start_coords_ors[0]},{start_coords_ors[1]}",
    "end": f"{end_coords_ors[0]},{end_coords_ors[1]}"
}

The request headers carry the API key:


In [ ]:
headers_directions_ors = {
    "Authorization": ors_api_key
}

##### 2.2.1.2. Sending the Request

Send a GET request to retrieve the route.


In [ ]:
response_directions_ors = requests.get(url_directions_ors, params=params_directions_ors, headers=headers_directions_ors)

##### 2.2.1.3. Checking the Response Status

Before processing the response, let's confirm that the request succeeded.


In [ ]:
response_directions_ors.status_code

If the status is `200`, proceed to parse the result. Any other code indicates an error — inspect the response body to understand what went wrong.


##### 2.2.1.4. Parsing the Response

The server returns a GeoJSON response.


In [ ]:
data_directions_ors = response_directions_ors.json()

The top-level keys again, to see how this response is organised. The full schema is described in the API documentation.


In [ ]:
data_directions_ors.keys()

Route information is stored under the `features` key.
Since we requested a single route, we access the first element.


In [ ]:
route_ors = data_directions_ors["features"][0]

route_ors.keys()

And what is stored in `properties`:


In [ ]:
properties_ors = route_ors["properties"]

properties_ors.keys()

The `summary` key contains overall route information (distance and travel time), while `segments` provides a more detailed breakdown of individual route legs.


Pull out the route distance and duration.


In [ ]:
distance_ors = properties_ors["summary"]["distance"]
duration_ors  = properties_ors["summary"]["duration"]

print(f"Distance (metres): {distance_ors}")
print(f"Duration (seconds): {duration_ors}")

##### 2.2.1.5. Creating a GeoDataFrame

Now a `GeoDataFrame` for the route.

Since the response is already in GeoJSON format, it can be converted to a `GeoDataFrame` directly without any additional geometry processing.


In [ ]:
route_gdf_ors = gpd.GeoDataFrame.from_features(
    data_directions_ors["features"], crs="EPSG:4326"
)

And one for the start and end points:


In [ ]:
points_gdf_ors = gpd.GeoDataFrame(
    {"name": ["Start", "End"]},
    geometry=gpd.points_from_xy(
        [start_coords_ors[0], end_coords_ors[0]],
        [start_coords_ors[1], end_coords_ors[1]]
    ),
    crs="EPSG:4326"
)

##### 2.2.1.6. Visualising the Result

The route and its endpoints on an interactive map:


In [ ]:
m = route_gdf_ors.explore(
    tiles="cartodbpositron",
    color="#FCDD9D",
    style_kwds={"weight": 5},
)

points_gdf_ors.explore(
    m=m,
    color="#C4C3E3",
    marker_kwds={"radius": 6},
    tooltip="name"
)


#### 2.2.2. Distance Matrix

A distance matrix returns the distances and travel times between multiple locations.

The endpoint is:

```python
/v2/matrix/{profile}
```

where:

- `profile` — the transport mode;
- `locations` — the list of point coordinates;
- `metrics` — the metrics to return (`distance`, `duration`).


##### 2.2.2.1. Building the Request Parameters

The endpoint for matrix calculation:


In [ ]:
url_matrix_ors = "https://api.openrouteservice.org/v2/matrix/driving-car"

Now the list of locations. OpenRouteService expects coordinates in [longitude, latitude] order.


In [ ]:
locations_coords_ors = [
    [16.3725, 48.2082],   # Stephansplatz
    [16.3122, 48.1847],   # Schönbrunn Palace
    [16.3956, 48.2166],   # Wiener Riesenrad
    [16.3806, 48.1917]    # Belvedere
]

The request body — what goes to the server with the POST request:

- `locations` — the list of coordinate points;
- `metrics` — the metrics to return.


In [ ]:
params_matrix_ors = {
    "locations": locations_coords_ors,
    "metrics": ["distance"]
}

##### 2.2.2.2. Request Headers

The API key must be included in the request headers.


In [ ]:
headers_matrix_ors = {
    "Authorization": ors_api_key,
    "Content-Type": "application/json"
}

##### 2.2.2.3. Sending the Request

The matrix endpoint requires a POST request (as specified in the documentation).


In [ ]:
response_matrix_ors = requests.post(url_matrix_ors, json=params_matrix_ors, headers=headers_matrix_ors)

##### 2.2.2.4. Checking the Response Status

Before processing the response, confirm that the request succeeded.


In [ ]:
response_matrix_ors.status_code

If the status is `200`, proceed to parse the result. Any other code indicates an error — inspect the response body to understand what went wrong.


##### 2.2.2.5. Parsing the Response

The server returns a JSON response, which can be converted to a Python dictionary using `.json()`.


In [ ]:
data_matrix_ors = response_matrix_ors.json()

The top-level keys:


In [ ]:
data_matrix_ors.keys()

The distance matrix (in metres) is stored under the `distances` key.


##### 2.2.2.6. Extracting the Data


Pull the distance matrix out of the response.


In [ ]:
distances_matrix_ors = data_matrix_ors["distances"]

print(f"Distance matrix (metres):\n{distances_matrix_ors}")

The result is a nested list (a matrix), where each row and column corresponds to one of the input locations.


#### 2.2.3. Isochrones


OpenRouteService can generate **isochrones** — zones reachable from a given point within a specified time or distance.

The endpoint is:

```python
/v2/isochrones/{profile}
```

where:

- `profile` — the transport mode (e.g. `foot-walking`, `driving-car`);
- the request body contains the origin point and the time or distance intervals.

In this example, we will generate walking isochrones for 5, 10, and 15 minutes.


##### 2.2.3.1. Building the Request Parameters

The endpoint:


In [ ]:
url_isochrones_ors = "https://api.openrouteservice.org/v2/isochrones/foot-walking"

The request body:


In [ ]:
start_coords_isochrones_ors = [16.3700, 48.2005]   # Karlsplatz

params_isochrones_ors = {
    "locations": [start_coords_isochrones_ors],
    "range": [300, 600, 900]
}

Here:

- `locations` — the origin point(s) from which isochrones are generated;
- `range` — time intervals in seconds:
  - `300` = 5 minutes,
  - `600` = 10 minutes,
  - `900` = 15 minutes.


And the headers with the API key:


In [ ]:
headers_isochrones_ors = {
    "Authorization": ors_api_key,
    "Content-Type": "application/json"
}

##### 2.2.3.2. Sending the Request


The isochrone endpoint requires a POST request.


In [ ]:
response_isochrones_ors = requests.post(url_isochrones_ors, json=params_isochrones_ors, headers=headers_isochrones_ors)

##### 2.2.3.3. Checking the Response Status


Before processing the response, confirm that the request succeeded.


In [ ]:
response_isochrones_ors.status_code

If the status is `200`, proceed to parse the result. Any other code indicates an error — inspect the response body to understand what went wrong.


##### 2.2.3.4. Parsing the Response

The server returns a **GeoJSON** response.


In [ ]:
data_isochrones_ors = response_isochrones_ors.json()

The top-level keys:


In [ ]:
data_isochrones_ors.keys()

The isochrone data is stored under `features`, where each isochrone is represented as a separate feature.


In [ ]:
data_isochrones_ors["features"]

Each feature contains:

- a geometry (the isochrone polygon);
- properties, including the time value for which the zone was generated.


##### 2.2.3.5. Converting to a GeoDataFrame

Since the response is already in GeoJSON format, it can be converted to a `GeoDataFrame` directly.


In [ ]:
isochrones_gdf_ors = gpd.GeoDataFrame.from_features(
    data_isochrones_ors["features"], crs="EPSG:4326"
)

##### 2.2.3.6. Visualising the Result

The isochrones on an interactive map:


In [ ]:
isochrones_gdf_ors.explore(tiles="cartodbpositron")

In this subsection, we covered a few examples of working with OpenRouteService — routing, distance matrices, and isochrones. The API supports many more features and additional request parameters; we encourage you to explore the documentation.


### 2.3. GraphHopper

**GraphHopper** is a routing and network analysis service built on OpenStreetMap data.
It supports routing, distance matrix calculation, and other transport network analysis tasks.

GraphHopper requires an **API key**, which you can obtain by registering on the service website.

[Documentation](https://docs.graphhopper.com/)

> Note: the cells in this section will not produce output automatically, as they require a personal API key. You can obtain one from your GraphHopper dashboard and run the code locally. The placeholder for the key is marked as `your_graph_key`.


#### 2.3.0. Storing the API Key


Before getting started, obtain an API key from [GraphHopper](https://www.graphhopper.com/) by registering on the site and copying the key from your dashboard.

Store the key in a variable.


In [ ]:
graphhopper_api_key = "your_graph_key"

#### 2.3.1. Shortest Route

GraphHopper can return a route between two or more points.

The endpoint is:

```python
/api/1/route
```

The request parameters are:

- `point` — route waypoints in `latitude,longitude` format;
- `profile` — the transport mode (e.g. `car`, `foot`, `bike`);
- `key` — the API key.

##### 2.3.1.1. Building the Request Parameters

The endpoint:


In [ ]:
url_directions_graph = "https://graphhopper.com/api/1/route"

The start and end points:


In [ ]:
# Coordinates (longitude, latitude)
start_coords_graph = [16.3755, 48.1855]  # Wien Hauptbahnhof
end_coords_graph = [16.3725, 48.2082]    # Stephansplatz

# Format as latitude,longitude strings (GraphHopper's expected order)
start_graph = f"{start_coords_graph[1]},{start_coords_graph[0]}"
end_graph = f"{end_coords_graph[1]},{end_coords_graph[0]}"

params_directions_graph = {
    "point": [start_graph, end_graph],
    "profile": "car",
    "key": graphhopper_api_key
}

Note that GraphHopper expects coordinates in the `point` parameter in **latitude,longitude** order — the opposite of some other services.


##### 2.3.1.2. Sending the Request

Send a **GET request** to the API.


In [ ]:
response_directions_graph = requests.get(url_directions_graph, params=params_directions_graph)

##### 2.3.1.3. Checking the Response Status

Before processing the response, confirm that the request succeeded.

In [ ]:
response_directions_graph.status_code

If the status is `200`, proceed to parse the result. Any other code indicates an error — inspect the response body to understand what went wrong.


##### 2.3.1.4. Parsing the Response

The server returns a JSON response:


In [ ]:
data_directions_graph = response_directions_graph.json()

The top-level keys:


In [ ]:
data_directions_graph.keys()

Route information is stored under the `paths` key as a list of route objects. Since we requested a single route, we access the first element.


In [ ]:
route_graph = data_directions_graph["paths"][0]

route_graph.keys()

- `distance` — route length in metres;
- `time` — travel time in milliseconds;
- `points` — route geometry as an encoded polyline.


Pull out the key route properties.


In [ ]:
distance_graph = route_graph["distance"]
duration_graph = route_graph["time"] / 1000
geometry_graph = route_graph["points"]

print(f"Distance (metres): {distance_graph}")
print(f"Duration (seconds): {duration_graph}")
print(f"Geometry: {geometry_graph[:50]}...")

The route geometry is returned as an encoded polyline to reduce response size. To use it for visualisation, it must first be decoded into a list of coordinates.


##### 2.3.1.5. Decoding the Geometry


Decode the geometry with the `polyline` library, as before.


In [ ]:
decoded_route_graph = polyline.decode(geometry_graph)

The result is a list of coordinate pairs describing the route. Let's print the first few:


In [ ]:
decoded_route_graph[:5]

Each point is returned as (latitude, longitude), since `polyline.decode()` uses that order.

`shapely` expects (longitude, latitude), so we need to swap the coordinates before creating the geometry.


In [ ]:
route_line_graph = LineString([(lon, lat) for lat, lon in decoded_route_graph])

##### 2.3.1.6. Creating a GeoDataFrame

A `GeoDataFrame` for the route:


In [ ]:
route_gdf_graph = gpd.GeoDataFrame(
    {"name": ["Route"]},
    geometry=[route_line_graph],
    crs="EPSG:4326"
)

And one for the start and end points:


In [ ]:
points_gdf_graph = gpd.GeoDataFrame(
    {"name": ["Start", "End"]},
    geometry=gpd.points_from_xy(
        [start_coords_graph[0], end_coords_graph[0]],
        [start_coords_graph[1], end_coords_graph[1]]
    ),
    crs="EPSG:4326"
)

##### 2.3.1.7. Visualising the Result

The route and its endpoints on an interactive map:


In [ ]:
m = route_gdf_graph.explore(
    tiles="cartodbpositron",
    color="#FCDD9D",
    style_kwds={"weight": 5},
    tooltip="name"
)

points_gdf_graph.explore(
    m=m,
    color="#504E76",
    marker_kwds={"radius": 6},
    tooltip="name"
)

In this example, we covered the basic GraphHopper routing workflow. The service also supports other network analysis methods, including distance matrix calculation — these are described in the GraphHopper documentation.


## Summary


In this section, we covered the fundamentals of **network analysis using external APIs**.

We learned:

- what an API is and how HTTP requests to web services are structured;
- how to read documentation and identify the required request parameters;
- how to send requests from Python using the `requests` library;
- how to parse responses and convert API results into `GeoDataFrame` objects for further analysis and visualisation.

External APIs are a convenient alternative to building and maintaining a local graph, as the computations are handled server-side on demand.

It is worth remembering that different services differ in request format, available features, rate limits, and authentication requirements. Reading the documentation carefully and understanding the response structure are essential when working with any API.
